# Module 6: Give the Bot a Library: Files, Chunking & Storage

A real document is far too long to paste into one prompt. So instead of cramming the whole thing in, we:

1. **Load** the file from disk,
2. **Chunk** it into small overlapping pieces,
3. **Embed** each piece into a vector (Week 5),
4. **Store** the vectors, and
5. **Query** the store to pull back only the few pieces that answer the question, and
6. **Audit the data boundary** where document chunks enter the embedding model.


## 0. Setup

Run this once in a terminal from inside your `rag-starter/` folder, then select
**`rag-starter/.venv`** as the notebook kernel:

```bash
uv venv && uv pip install -e '.[dev]'
```

The editable install makes `ribot` importable from any notebook location. The next cell also
gets the repository root from the installed package so this lesson can load the sample documents.

In [1]:
from pathlib import Path
# GIVEN — run as-is. Confirm that this notebook kernel can import the course package.
try:
    import ribot
    print("✓ Connected to the editable ribot package.")
except ModuleNotFoundError:
    print("ribot is not installed in this kernel — using local fallback definitions below.")

REPO_ROOT = Path(ribot.__file__).resolve().parent.parent
print("rag-starter at:", REPO_ROOT)
print("has ribot/:", (REPO_ROOT / "ribot").is_dir())

✓ Connected to the editable ribot package.
rag-starter at: /Users/luyaoniu/Desktop/Teaching-UW/HMSTI 509/2026/rag-starter
has ribot/: True


## 1. Load a file (with missing-file handling)

A `.md` or `.txt` file is just text on disk. We read it inside a `with` block so Python closes the file for us automatically — even if something goes wrong.

**Why `with`?** It guarantees the file handle is released. You never have to remember to call `.close()`.

**Why handle the missing file?** A typo in a path should print a friendly message, not crash your whole pipeline. We catch `OSError` (covers missing files *and* permission problems).

In [2]:
def load_text(path):
    """Read a text file safely. Returns the text, or None if the file can't be read."""
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except OSError as err:
        print(f"could not read {path}: {err}")
        return None


SAMPLE_DIR = REPO_ROOT / "data" / "sample_docs"
help_text = load_text(SAMPLE_DIR / "help_center.md")

print("characters loaded:", len(help_text))
print("--- first 200 chars ---")
print(help_text[:200])

characters loaded: 964
--- first 200 chars ---
# Acme Cloud — Help Center (synthetic sample data)

## Resetting your password
If you forgot your password, click "Forgot password" on the sign-in page and enter the
email on your account. You will re


## 2. Chunk the text (and why overlap matters)

**Why chunk at all?** Our sample docs are tiny, but a real handbook can run 100,000+ characters. Embedding whole documents causes three problems:

1. **One vector blurs many topics.** A doc about passwords, billing, and lockouts gets one averaged vector that's close to none of them.
2. **The prompt has a budget.** We want to hand the LLM only the relevant paragraph, not the whole handbook.
3. **Citations need to be specific.** "See chunk 14" is checkable; "see the handbook" isn't.

So we slice the text into small **chunks**. Too small, though, and a chunk loses its context — you'll tune that in Your turn #3.

`chunk_text(text, size, overlap)` lives in `ribot/chunking.py`. It walks the text in steps of `size - overlap`, so consecutive chunks **share** `overlap` characters.

**Why overlap?** If a key sentence straddles a chunk boundary, a clean cut would split the idea in half and *neither* chunk would match well. Overlap keeps the whole idea intact in at least one chunk.

In [ ]:
from ribot.chunking import chunk_text

# Tiny example so you can SEE the overlap. size=4, overlap=1:
demo = chunk_text("abcdefghij", size=4, overlap=1)
print(demo)
# Notice each chunk shares its last char with the next chunk's first char:
# 'abcd' -> 'defg' -> 'ghij' -> 'j' (the 'd' and 'g' are repeated).

['abcd', 'defg', 'ghij', 'j']


### ✏️ Your turn #1 — predict the chunks before you run

Work it out on paper first: `chunk_text("retrievalrocks", size=5, overlap=2)` starts a new
chunk every `size - overlap = 3` characters (positions 0, 3, 6, …). **How many chunks come
back, and what exactly is the second one?** Write your predictions below, *then* run the cell
to check yourself.

In [ ]:
predicted_count = 0         # <- your prediction: how many chunks?
predicted_second = "?????"  # <- your prediction: the exact 5 characters of chunk #2

actual = chunk_text("retrievalrocks", size=5, overlap=2)
print("actual chunks:", actual)
print("count  — you said", predicted_count, "| actual:", len(actual))
print("second — you said", repr(predicted_second), "| actual:", repr(actual[1]))

if predicted_count == len(actual) and predicted_second == actual[1]:
    print("PASS  you can see the overlap machinery ✓")
else:
    print("HINT  chunks start at positions 0, 3, 6, 9, 12 — and the last one may be shorter than size")

In [ ]:
# Now chunk the real document with the default RAG sizes.
chunks = chunk_text(help_text, size=300, overlap=50)
print(f"{len(chunks)} chunks from help_center.md\n")
for i, c in enumerate(chunks):
    print(f"[chunk {i}] {c[:70]!r}...")

## 3. Embed each chunk

`get_embedder()` returns the **real** embedder if you have an API key, or the offline `fallback_embed()` if you don't. Either way, it turns a chunk of text into a vector (a list of numbers).

We stay fully offline here: no key means `fallback_embed()`, a deterministic hashed bag-of-words. It's not as good as a real model, but it's fine for learning the pipeline.

In [ ]:
from ribot.embeddings import get_embedder

embed = get_embedder()

vec = embed(chunks[0])
print("embedding length:", len(vec))
print("first 5 numbers:", [round(x, 3) for x in vec[:5]])

### Data-boundary check: where did that chunk go?

You just passed `chunks[0]` to `embed(...)`. That is the moment document text either stays on this machine or crosses an API boundary. 

Run the audit cell below before indexing any real corpus. 

In [ ]:
from ribot.config import get_settings

settings = get_settings()
if settings.embed_backend == "ollama":
    route = f"LOCAL MODEL via Ollama ({settings.ollama_embed_model})"
elif settings.has_key:
    route = f"API MODEL ({settings.embed_model}) — chunk text crosses the configured API boundary"
else:
    route = "OFFLINE TEACHING FALLBACK — local word counter, not a trained model"

print("Current embedding route:", route)
print("Current chat route:     ", "LOCAL via Ollama" if settings.llm_backend == "ollama" else "default FakeLLM/API route")
print("Decision to record: Is this configuration appropriate for the documents I am indexing? Why?")

## 4. Store the chunks in a vector store

A **vector store** keeps chunks + their vectors together and can rank them by similarity.

We use `InMemoryStore` — pure Python, no install, lives in `ribot/vectorstore.py`. Its `add()` takes three parallel lists: `ids`, `documents` (the chunk text), and `embeddings` (the vectors).

> **The persistent "real" path:** `ChromaStore` saves vectors to disk so they survive a restart. `get_store(persist=True)` returns Chroma if it's installed, else falls back to in-memory. We mention it here, but everything in this lesson uses the offline in-memory store.

In [ ]:
from ribot.vectorstore import InMemoryStore

store = InMemoryStore()

ids = []
documents = []
embeddings = []
for i, chunk in enumerate(chunks):
    ids.append(f"help_center.md:{i}")
    documents.append(chunk)
    embeddings.append(embed(chunk))

store.add(ids=ids, documents=documents, embeddings=embeddings)

# Self-check card: count() should equal the number of chunks we added.
print("stored chunks:", store.count())
assert store.count() == len(chunks), "count should match the number of chunks!"


### Look inside the store

A vector store is not magic: `InMemoryStore` is just three parallel lists, `store.ids`, `store.documents`, and `store.embeddings`. Position `i` in each list belongs to the same chunk. Print them to see exactly what got saved.

In [ ]:
# Peek inside: every entry is (id, chunk text, vector), lined up by position.
print(f"{store.count()} entries | vector length: {len(store.embeddings[0])}\n")

for i, (chunk_id, doc, vec) in enumerate(zip(store.ids, store.documents, store.embeddings)):
    nonzero = sum(1 for x in vec if x != 0)
    print(f"[{i}] id: {chunk_id}")
    print(f"    text ({len(doc)} chars): {doc[:120]!r}...")
    print(f"    vector: {[round(x, 3) for x in vec[:8]]} ... ({nonzero} non-zero of {len(vec)})")
    print()

## 5. Query the store — the payoff

Now ask a question. `store.query(vector, n_results=3)` embeds *nothing for you* — we hand it the **question's** vector, and it returns the top-N `(chunk, score)` pairs.

That `score` is exactly the **cosine similarity** between the question vector and each chunk vector. Higher = more similar meaning. The store just does it for every chunk and sorts.

### Which file did that chunk come from?

`query()` hands back the chunk **text** and a score — but not the filename. And a chunk without
a source is not much use: *"the handbook says 15 days"* is only trustworthy if you can say
**which** handbook, and go read it.

That's what the `ids` list is for. Look back at section 4: we stored each chunk under an id
shaped `"help_center.md:2"` — **filename, colon, chunk number**. So the source is always
recoverable:

1. find where this chunk sits in `store.documents`,
2. read the id at the same position,
3. take the part before the `:`.

`source_of()` below does exactly that. **This is the piece Week 7 needs**: `reply_grounded()`
labels every retrieved chunk `[source: file.md]` so the answer can cite it, and Week 8's agent
does the same. The reference version lives in `ribot/retriever.py` as
`Retriever.source_of(chunk)`.

In [ ]:
def source_of(store, chunk):
    """Which file did this chunk come from? id 'help_center.md:2' -> 'help_center.md'."""
    position = store.documents.index(chunk)     # where this chunk sits in the list
    chunk_id = store.ids[position]              # the id stored at the same position
    return chunk_id.split(":")[0]               # everything before the colon


def ask(store, question, k=3):
    """Embed the question, return the top-k (chunk, score) matches, and print them with sources."""
    q_vec = embed(question)
    hits = store.query(q_vec, n_results=k)
    print(f"Q: {question}\n")
    for rank, (chunk, score) in enumerate(hits, start=1):
        print(f"  #{rank}  [score {score:.3f}]  [source: {source_of(store, chunk)}]")
        print(f"       {chunk[:90]!r}...")
    return hits


_ = ask(store, "How do I reset my password?")

In [ ]:
# Try another question. The top chunk should shift to the matching section.
_ = ask(store, "What happens when my account is locked?")

### ✏️ Your turn #2 — one question it CAN answer, one it CAN'T

Retrieval always returns *something* — the real question is whether the top score is worth
trusting. Ask the store one question the help-center docs genuinely answer and one they
couldn't possibly, then compare the top scores. This eyeball test is exactly what your
assignment's golden set turns into a number (**hits@3**).

In [ ]:
answerable   = "How do I turn on two-factor authentication?"   # <- your own question the docs DO cover
unanswerable = "What is the moon made of?"           # <- plausible for a user to ask, but NOT in the docs

hits_yes = ask(store, answerable)
hits_no = ask(store, unanswerable)

gap = hits_yes[0][1] - hits_no[0][1]
print(f"top score, answerable:   {hits_yes[0][1]:.3f}")
print(f"top score, unanswerable: {hits_no[0][1]:.3f}")
print(f"gap: {gap:.3f}  (a healthy store separates signal from noise)")
if unanswerable == "What is the moon made of?":
    print("HINT  write your OWN unanswerable question — something a real user of your bot might actually ask")

## 6. "My top chunk looks wrong" — self-diagnosis checklist

When retrieval returns something irrelevant, walk this checklist **before** asking for help:

1. **Is the store empty?** Check `store.count()`. If it's 0, your `add()` step didn't run or the loop appended nothing.
2. **Did you load the right file?** Print the first 200 chars of the loaded text. A typo'd path may have returned `None`.
3. **Are your chunks a sane size?** `print(len(chunks))` and eyeball one. Huge chunks dilute the match; tiny chunks lose context. Start with `size=300, overlap=50`.
4. **Does your question share words with the docs?** The offline `fallback_embed()` matches on shared words. A question phrased nothing like the source text will score low — try wording it closer to how the doc talks.
5. **Did you embed the question with the SAME embedder you used for the chunks?** Mixing embedders gives nonsense scores.

> **Two failures.** Notice that "looks wrong" covers two distinct
> things: (a) the chunk that *should* have won never came back at all, and (b) an irrelevant chunk
> came back at the top. Machine learning calls those a **recall** failure and a **precision**
> failure, and they are the same pair used to grade every classifier ever built. They are worth
> separating because they have different fixes: a recall failure usually means your chunks are wrong
> (too big, too small, split across the boundary that held the answer); a precision failure usually
> means your *scoring* is wrong (a keyword-matching embedder being fooled by shared words). The
> `hits@3` number you are about to build measures recall — did the right chunk make the top 3.

In [ ]:
# Quick diagnostic you can paste any time retrieval looks off:
print("store.count():", store.count())
print("num chunks   :", len(chunks))
print("first chunk  :", documents[0][:80] if documents else "(none)")

## 7. The whole pipeline

The steps above are exactly what `ribot/build_index.py` does over a whole **folder** of documents. Here it is end-to-end — load every `.md`/`.txt` in the sample folder, chunk, embed, store, and query.

In [ ]:
def build_index(folder):
    """load -> chunk -> embed -> store, over every .md/.txt file in a folder."""
    store = InMemoryStore()
    ids, documents, embeddings = [], [], []
    for path in sorted(folder.glob("*")):
        if path.suffix.lower() not in {".md", ".txt"}:
            continue
        text = load_text(path)
        if text is None:
            continue
        for i, chunk in enumerate(chunk_text(text, size=300, overlap=50)):
            ids.append(f"{path.name}:{i}")
            documents.append(chunk)
            embeddings.append(embed(chunk))
    store.add(ids=ids, documents=documents, embeddings=embeddings)
    print(f"indexed {store.count()} chunks from {folder.name}/")
    return store


full_store = build_index(SAMPLE_DIR)
_ = ask(full_store, "How much does the Team plan cost?")

### ✏️ Your turn #3 — tune the chunking knobs

`size=300, overlap=50` is a starting point (unlikely to be optimal). Re-run the pipeline at three sizes and
watch what happens to the chunk count, the top score, and how *readable*
the winning chunk is. Then answer the two questions below.

In [ ]:
question = "How do I reset my password?"   # <- after the first run, swap in a question of yours

for size, overlap in [(80, 10), (300, 50), (1200, 100)]:
    s = InMemoryStore()
    ids, docs, embs = [], [], []
    for i, chunk in enumerate(chunk_text(help_text, size=size, overlap=overlap)):
        ids.append(f"help:{i}")
        docs.append(chunk)
        embs.append(embed(chunk))
    s.add(ids=ids, documents=docs, embeddings=embs)
    top_chunk, top_score = s.query(embed(question), n_results=1)[0]
    print(f"size={size:>4}  chunks={s.count():>3}  top score={top_score:.3f}")
    print(f"      top chunk: {top_chunk[:80]!r}...")
    print()

**Answer in one sentence each:**

1. Which size scored highest — and is its winning chunk actually the most *useful* text to hand a model?
2. Why can a tiny chunk score high yet still produce a bad answer? *(Hint: what got cut off at its edges?)*

### ✏️ Your card — Week 6: the chunk-size ablation

You just ran three chunk sizes and read three different winning chunks. Changing one setting
while holding everything else still has a name — an **ablation** — and it is how every default
in a serious system gets chosen.

Right now your comparison is judged by eye. In your assignment you'll rerun it against your
**golden set** so the judgment is a number (hits@3) instead of a vibe. Start the card here:

> **What your golden set actually is — and the trap built into it.** Those questions are a
> **held-out set**: written from what the documents *should* be able to answer, not tuned to what
> your index happens to return. That is the same discipline machine learning calls train/validation/
> test, and it is the only reason the number means anything.
>
> Here is the trap. You are about to try several chunk sizes and keep the one with the best `hits@3`
> — measured on **the same ten questions every time**. Do that often enough and you stop tuning the
> index and start tuning it *to those ten questions*. That is **overfitting to your eval set**, and
> it is why ML keeps a validation set (which you are allowed to tune against) separate from a test
> set (which you look at once, at the end). You do not need a second set this week — you need to
> know the trap exists, and to be suspicious of a score you improved by trying twenty things. Week 7
> gives you the honest check: extend the set and see whether the number survives.

- **Hypothesis:** smaller chunks will ____ retrieval quality, because ____
- **Change:** `size` (and `overlap`) in `chunk_text`.
- **Held fixed:** the corpus, the embedder, the question(s), the store.
- **Metric & prediction:** top score and the readability of the winning chunk here; **hits@3**
  on your golden set in the assignment. I predict ____
- **Result & decision:** ____ *(name the size you'll carry into Week 7 — and the number that made you pick it)*

**Then read your failures, not just your score.** Take the **2 worst results** from your golden
set — a miss, or the lowest-scoring hit — print the top-3 chunks each one actually returned,
and write one sentence per miss diagnosing *why* the wrong chunk won: keyword overlap? the
answer split across a chunk boundary? a vague question? That's **error analysis**, and it is
the difference between knowing your score and knowing what to fix.


## You shipped the retrieval half of RAG

You now have a searchable **library**: load → chunk → embed → store → query, all offline.

- `chunk_text(size, overlap)` — overlap keeps split ideas whole.
- `get_embedder()` — the Week 5 embedder, reused.
- `InMemoryStore` + `query()` — cosine similarity over many chunks.
- `source_of(store, chunk)` — the id trick that turns a chunk back into a **filename**. This
  is what makes a *citation* possible, and Week 7 depends on it.
- The "my top chunk looks wrong" checklist — self-diagnose before asking.

**Next:** the assignment has you build this for your **own** corpus, commit `build_index.py`,
and turn "looks right" into a number — a 10-question **golden set** (`eval_questions.json`)
plus `evaluate_retrieval()` printing **hits@3**. In **Week 7** we wire this library to the
bot's generation so it answers grounded, cited questions — `reply_grounded()` is your `ask()`
plus `source_of()`, with the chunks pasted into the prompt — and wrap that golden set as a
pytest.